# 01 · 最小的工具调用循环

**想弄清楚的问题：** Agent 是怎样调用工具的，又是怎样决定停下来的？

这本 notebook 只有一个工具（计算器），不用任何 Agent 框架。全部逻辑分四块，每块一个单元格：

1. **模型调用**：把对话交给模型，拿回一段文字
2. **工具执行**：程序（不是模型）真正去算
3. **上下文组装**：把工具结果作为一条新消息放回对话
4. **循环**：重复 1–3，直到模型说“我答完了”或步数用完

模型是没有微调过的 Qwen2.5-0.5B（4-bit，本机运行）。循环只要求 `call_model(messages) -> str`，
换成任何托管模型的 API 只需要重写第 1 块。

运行前先在 `tiny-agent/` 目录执行 `sh run.sh setup` 和 `sh run.sh download --model qwen2.5`。

## 1. 模型调用

模型只做一件事：读完整段对话，续写下一段文字。它不会执行任何东西。

In [1]:
from pathlib import Path

from mlx_lm import generate, load
from mlx_lm.sample_utils import make_sampler

model, tokenizer = load(str(Path("../models/qwen2.5-0.5b-4bit")))
greedy = make_sampler(temp=0.0)  # 每一步都选分数最高的词，同样的输入永远得到同样的输出


def call_model(messages):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return generate(model, tokenizer, prompt, max_tokens=100, sampler=greedy).strip()


call_model([{"role": "user", "content": "用一句话介绍你自己"}])

'我是来自阿里云的大规模语言模型，我叫通义千问。'

## 2. 工具执行

工具就是一个普通的 Python 函数。这里故意不用 `eval`：模型给的字符串不可信，只允许数字、括号和四则运算。

In [2]:
import ast
import operator

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}


def calculator(expression):
    def visit(node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](visit(node.left), visit(node.right))
        raise ValueError("只支持数字、括号和 + - * /")

    return format(visit(ast.parse(expression, mode="eval").body), ".12g")


calculator("19*23"), calculator("(19+23)*2")

('437', '84')

## 3. 上下文组装，4. 循环

约定：模型每次只输出一个 JSON。`{"tool": ...}` 表示“请替我调用工具”，`{"final": ...}` 表示“我答完了”。

**循环什么时候停？** 只有两种情况：模型输出了 `final`，或者步数用完。
模型自己并不知道工具有没有被执行，它只是看到对话里多了一条“工具结果”。

In [3]:
import json

SYSTEM = """你是一个助手，每次只输出一个 JSON 对象，不要输出别的文字。
需要计算时调用计算器：{"tool":"calculator","args":{"expression":"12*3"}}
收到工具结果后给出最终回答：{"final":"回答内容"}
不需要计算时直接输出 final。"""


def run(question, max_steps=4):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        raw = call_model(messages)                                   # 1. 模型调用
        print(f"[{step}] 模型  → {raw}")
        messages.append({"role": "assistant", "content": raw})
        try:
            action = json.loads(raw)
            if "final" in action:                                    # 停止条件一：模型说答完了
                return action["final"]
            result = calculator(action["args"]["expression"])        # 2. 工具执行
            print(f"    工具  → {result}")
            messages.append({"role": "user", "content": "工具结果：" + result})   # 3. 上下文组装
        except Exception as exc:                                     # 格式不对、工具报错：告诉模型，让它重试
            print(f"    出错  → {exc!r}")
            messages.append({"role": "user", "content": f"出错了：{exc}。请重新输出正确的 JSON。"})
    return None                                                      # 停止条件二：步数用完

## 一次成功的运行

In [4]:
run("请计算 19*23")

[1] 模型  → {"tool":"calculator","args":{"expression":"19*23"}}
    工具  → 437
[2] 模型  → {"final":"437"}


'437'

模型第一步提出调用，程序算出 437，模型第二步看到结果后给出 `final`。`19*23 = 437`，正确。

## 一次失败的运行

In [5]:
run("(19+23)*2 等于多少？")

[1] 模型  → {"tool":"calculator","args":{"expression":"19+23*2"}}
    工具  → 65
[2] 模型  → {"final":"65"}


'65'

正确答案是 **84**，这里得到 **65**。

值得注意的是失败发生在哪里：

- 循环**正常结束**了：模型调用了工具，工具没有报错，模型给出了 `final`。
- 工具也**算对了**：`19+23*2` 确实等于 65。
- 错在模型把问题抄成算式时**丢了括号**。

所以“循环正常结束”和“答案正确”是两回事。程序这一侧没有任何信号能发现这个错误，除非另外拿正确答案来核对。

## 另一种失败：不需要工具的问题

In [6]:
run("你好，你是谁？")

[1] 模型  → {"tool":"chatbot","args":{"name":"小明","age":25,"interests":["阅读","编程"]}}
    出错  → KeyError('expression')
[2] 模型  → {"tool":"calculator","args":{"expression":"12*3"}}
    工具  → 36


[3] 模型  → {"final":"出错了：'expression'。请重新输出正确的 JSON。"}


"出错了：'expression'。请重新输出正确的 JSON。"

这次模型编了一个不存在的工具 `chatbot`，收到报错后又照抄了提示词里的示例 `12*3`，
最后把报错信息当成了回答。提示词里唯一的示例是一次工具调用，小模型倾向于模仿它，哪怕问题根本不需要工具。

## 我学到了什么

1. **模型只产生文字，程序才执行。** “调用工具”其实是模型写出一段符合约定的 JSON，由循环解析并执行。
2. **停止是循环的规则，不是模型的能力。** 模型输出 `final` 或步数用完，循环才停；去掉步数上限，第三个例子可能一直转下去。
3. **上下文就是全部记忆。** 模型能“知道”工具结果，只因为循环把它追加成了一条消息。
4. **正常结束不等于做对了。** 第二个例子每一步都没报错，答案却是错的。
5. **最先坏掉的是“要不要用工具”和“参数抄得对不对”，不是循环本身。**

## 下一个问题

提示词里怎样描述工具，会多大程度上改变模型的行为？第三个例子提示答案是“很大”。
这已经超出“试试看”的范围，需要固定其他条件、只改这一处、数清楚对错，
所以放在 `docs/experiment-tool-wording.md` 里作为一次对照实验。